# Exploration of the dataset

## Imports

In [6]:
import sys
sys.path.append("../src")
import os
import rasterio
import numpy as np
from dataset import check_all_patches_have_all_files

## Verification of missing files

We first verify if for each .tif files we have:
- a simple .tif
- a _cl.tif
- a _conf.tif


In [3]:
check_all_patches_have_all_files("../data/raw/MARIDA/patches")

Checking folder: S2_30-8-18_16PCC
  Found bases: ['S2_30-8-18_16PCC_0', 'S2_30-8-18_16PCC_1', 'S2_30-8-18_16PCC_10', 'S2_30-8-18_16PCC_11', 'S2_30-8-18_16PCC_12', 'S2_30-8-18_16PCC_13', 'S2_30-8-18_16PCC_14', 'S2_30-8-18_16PCC_15', 'S2_30-8-18_16PCC_16', 'S2_30-8-18_16PCC_17', 'S2_30-8-18_16PCC_18', 'S2_30-8-18_16PCC_19', 'S2_30-8-18_16PCC_2', 'S2_30-8-18_16PCC_20', 'S2_30-8-18_16PCC_21', 'S2_30-8-18_16PCC_22', 'S2_30-8-18_16PCC_23', 'S2_30-8-18_16PCC_24', 'S2_30-8-18_16PCC_25', 'S2_30-8-18_16PCC_26', 'S2_30-8-18_16PCC_27', 'S2_30-8-18_16PCC_28', 'S2_30-8-18_16PCC_29', 'S2_30-8-18_16PCC_3', 'S2_30-8-18_16PCC_30', 'S2_30-8-18_16PCC_31', 'S2_30-8-18_16PCC_32', 'S2_30-8-18_16PCC_33', 'S2_30-8-18_16PCC_34', 'S2_30-8-18_16PCC_35', 'S2_30-8-18_16PCC_36', 'S2_30-8-18_16PCC_37', 'S2_30-8-18_16PCC_38', 'S2_30-8-18_16PCC_39', 'S2_30-8-18_16PCC_4', 'S2_30-8-18_16PCC_40', 'S2_30-8-18_16PCC_41', 'S2_30-8-18_16PCC_42', 'S2_30-8-18_16PCC_43', 'S2_30-8-18_16PCC_44', 'S2_30-8-18_16PCC_45', 'S2_30-8-18_

There are no missing files we can proceed

## Inspection of a patch sample in detail

Now we inspect a tif

In [7]:
root = "../data/raw/MARIDA/patches/S2_1-12-19_48MYU"
base = "S2_1-12-19_48MYU_0"

img_path = os.path.join(root, base + ".tif")
mask_path = os.path.join(root, base + "_cl.tif")
conf_path = os.path.join(root, base + "_conf.tif")

with rasterio.open(img_path) as src:
    img = src.read()
    print("Image shape:", img.shape)
    print("dtype:", img.dtype)
    print("range:", img.min(), img.max())

with rasterio.open(mask_path) as src:
    mask = src.read(1)
    print("Mask unique classes:", np.unique(mask))

with rasterio.open(conf_path) as src:
    conf = src.read(1)
    print("Conf unique values:", np.unique(conf))

print("Shape alignment:", img.shape[1:], mask.shape, conf.shape)

Image shape: (11, 256, 256)
dtype: float32
range: 0.014371733 0.2712906
Mask unique classes: [ 0.  5.  7. 14.]
Conf unique values: [0. 1.]
Shape alignment: (256, 256) (256, 256) (256, 256)


From this first look at one file

We see that our tif image has a resolution of 256 x 256 pixels and 11 spectral bands. We will therefore have to make a model that handles those 11 bannds. We will also shape our tensors as follows [[11], [256], [256]].

The dtype being float means we do not need to parse values to other forms of variables.

The range tells us that each pixel seems to vary from a value of 0.0 to 1.0.

The mask unique classes confirm that the mask is not binary but has multiple classes which are allocated to different labels on the MARIDA website.

Confidence unique values tell use the confidence is binary. 0 being low confidence and 1 high confidence

Shape allignement tells us that the img, mask and confidence files each have the same shapes and therefore no treatment is needed to rectify this.



## Inspection of all patch files

We will first verify that the shape of all patch files are the same

In [22]:

def check_shape_consistency(root):
    print("Checking shape consistency across all patches...\n")

    all_good = True
    first_shape = None

    # Loop through each patch folder
    for patch_folder in os.listdir(root):
        folder_path = os.path.join(root, patch_folder)
        if not os.path.isdir(folder_path):
            continue

        print(f"Folder: {patch_folder}")

        # Loop through all base images (ignore _cl and _conf)
        for f in os.listdir(folder_path):
            if f.endswith(".tif") and "_cl" not in f and "_conf" not in f:
                base = f.replace(".tif", "")
                img_path = os.path.join(folder_path, base + ".tif")
                mask_path = os.path.join(folder_path, base + "_cl.tif")
                conf_path = os.path.join(folder_path, base + "_conf.tif")

                # Load image, mask, conf
                try:
                    with rasterio.open(img_path) as src:
                        img = src.read()
                    with rasterio.open(mask_path) as src:
                        mask = src.read(1)
                    with rasterio.open(conf_path) as src:
                        conf = src.read(1)
                except Exception as e:
                    print(f"  ERROR reading files for {base}: {e}")
                    all_good = False
                    continue

                # Get shapes
                img_shape = img.shape[1:]     # (H, W)
                mask_shape = mask.shape       # (H, W)
                conf_shape = conf.shape       # (H, W)

                # Print shape info
                print(f"  {base}: img={img_shape}, mask={mask_shape}, conf={conf_shape}")

                # Check mask and conf match image shape
                if img_shape != mask_shape or img_shape != conf_shape:
                    print(f"    SHAPE MISMATCH in {base}!")
                    all_good = False

                # Check global consistency across all files
                if first_shape is None:
                    first_shape = img_shape
                else:
                    if img_shape != first_shape:
                        print(f"    GLOBAL INCONSISTENCY: {base} has shape {img_shape}, expected {first_shape}")
                        all_good = False

        print("")  # blank line after folder

    # Final summary
    if all_good:
        print("\nAll images, masks, and confidence maps have consistent shapes. No issues found.")
    else:
        print("\nThere are shape inconsistencies or missing data. Review required.")


In [23]:
check_shape_consistency("../data/raw/MARIDA/patches")

Checking shape consistency across all patches...

Folder: S2_30-8-18_16PCC
  S2_30-8-18_16PCC_25: img=(256, 256), mask=(256, 256), conf=(256, 256)
  S2_30-8-18_16PCC_15: img=(256, 256), mask=(256, 256), conf=(256, 256)
  S2_30-8-18_16PCC_20: img=(256, 256), mask=(256, 256), conf=(256, 256)
  S2_30-8-18_16PCC_12: img=(256, 256), mask=(256, 256), conf=(256, 256)
  S2_30-8-18_16PCC_10: img=(256, 256), mask=(256, 256), conf=(256, 256)
  S2_30-8-18_16PCC_39: img=(256, 256), mask=(256, 256), conf=(256, 256)
  S2_30-8-18_16PCC_5: img=(256, 256), mask=(256, 256), conf=(256, 256)
  S2_30-8-18_16PCC_28: img=(256, 256), mask=(256, 256), conf=(256, 256)
  S2_30-8-18_16PCC_34: img=(256, 256), mask=(256, 256), conf=(256, 256)
  S2_30-8-18_16PCC_19: img=(256, 256), mask=(256, 256), conf=(256, 256)
  S2_30-8-18_16PCC_29: img=(256, 256), mask=(256, 256), conf=(256, 256)
  S2_30-8-18_16PCC_26: img=(256, 256), mask=(256, 256), conf=(256, 256)
  S2_30-8-18_16PCC_21: img=(256, 256), mask=(256, 256), conf=(

All shapes across patches are consistent

We now check dtype consistency accross all patches

In [30]:
def check_dtype_consistency(root):
    print("Checking dtype consistency across all patches...\n")

    img_dtypes = set()
    mask_dtypes = set()
    conf_dtypes = set()

    all_good = True

    for patch_folder in os.listdir(root):
        folder_path = os.path.join(root, patch_folder)
        if not os.path.isdir(folder_path):
            continue

        print(f"Folder: {patch_folder}")

        for f in os.listdir(folder_path):
            # main image
            if f.endswith(".tif") and "_cl" not in f and "_conf" not in f:
                path = os.path.join(folder_path, f)
                with rasterio.open(path) as src:
                    img_dtype = src.dtypes[0]   # dtype of first band
                    img_dtypes.add(img_dtype)
                    print(f"  {f}   img dtype: {img_dtype}")

            # mask
            if f.endswith("_cl.tif"):
                path = os.path.join(folder_path, f)
                with rasterio.open(path) as src:
                    mask_dtype = src.dtypes[0]
                    mask_dtypes.add(mask_dtype)
                    print(f"  {f}   mask dtype: {mask_dtype}")

            # confidence
            if f.endswith("_conf.tif"):
                path = os.path.join(folder_path, f)
                with rasterio.open(path) as src:
                    conf_dtype = src.dtypes[0]
                    conf_dtypes.add(conf_dtype)
                    print(f"  {f}   conf dtype: {conf_dtype}")

        print("")  # space between folders

    # Summary
    print("\nSummary of dtypes found:")
    print("  Image dtypes:", img_dtypes)
    print("  Mask dtypes: ", mask_dtypes)
    print("  Conf dtypes: ", conf_dtypes)

    if len(img_dtypes) == 1 and len(mask_dtypes) == 1 and len(conf_dtypes) == 1:
        print("\nAll dtype checks passed. Everything is consistent.")
    else:
        print("\nWARNING: Inconsistent dtypes detected! Check above output.")

In [33]:
check_dtype_consistency("../data/raw/MARIDA/patches")

Checking dtype consistency across all patches...

Folder: S2_30-8-18_16PCC
  S2_30-8-18_16PCC_27_cl.tif   mask dtype: float32
  S2_30-8-18_16PCC_39_conf.tif   conf dtype: float32
  S2_30-8-18_16PCC_43_conf.tif   conf dtype: float32
  S2_30-8-18_16PCC_4_conf.tif   conf dtype: float32
  S2_30-8-18_16PCC_45_cl.tif   mask dtype: float32
  S2_30-8-18_16PCC_31_cl.tif   mask dtype: float32
  S2_30-8-18_16PCC_13_conf.tif   conf dtype: float32
  S2_30-8-18_16PCC_29_cl.tif   mask dtype: float32
  S2_30-8-18_16PCC_25.tif   img dtype: float32
  S2_30-8-18_16PCC_19_cl.tif   mask dtype: float32
  S2_30-8-18_16PCC_41_conf.tif   conf dtype: float32
  S2_30-8-18_16PCC_15.tif   img dtype: float32
  S2_30-8-18_16PCC_13_cl.tif   mask dtype: float32
  S2_30-8-18_16PCC_35_cl.tif   mask dtype: float32
  S2_30-8-18_16PCC_33_cl.tif   mask dtype: float32
  S2_30-8-18_16PCC_10_conf.tif   conf dtype: float32
  S2_30-8-18_16PCC_36_cl.tif   mask dtype: float32
  S2_30-8-18_16PCC_7_cl.tif   mask dtype: float32
  S2_

All files have consistent dtypes

We now check the range is indeed [0.0 ; 1.0]

In [26]:

def compute_range_for_folder(root):
    global_min = float("inf")
    global_max = float("-inf")

    patch_ranges = {}

    # Loop through each patch folder
    for patch_folder in os.listdir(root):
        folder_path = os.path.join(root, patch_folder)
        if not os.path.isdir(folder_path):
            continue

        patch_ranges[patch_folder] = []

        # Loop through all .tif files in the folder
        for f in os.listdir(folder_path):
            # Only examine the main image files: ignore _cl and _conf
            if f.endswith(".tif") and "_cl" not in f and "_conf" not in f:
                img_path = os.path.join(folder_path, f)

                # Load the image
                with rasterio.open(img_path) as src:
                    img = src.read()  # shape: (bands, H, W)

                local_min = img.min()
                local_max = img.max()

                patch_ranges[patch_folder].append((f, local_min, local_max))

                # Update global range
                global_min = min(global_min, local_min)
                global_max = max(global_max, local_max)



    return patch_ranges, global_min, global_max


In [29]:
patch_ranges, global_min, global_max = compute_range_for_folder("../data/raw/MARIDA/patches")
print(global_min, global_max)

-0.0144022 1.4251484


This is interesting, after cereful inspection of all patches we notice that the values do not range from 0.0 to 1.0 but from -0.0144022 to 1.4251484


This can be explained by the nature of the dataset which is already partly preprocessed. Although surface reflectance values are ideally expected to fall within the 0 to 1 range, it is normal here to observe slightly negative values and values above 1 due atmospheric correction algorithms applied to the raw Sentinel-2 data, and such algorithms can produce small negative numbers due to sensor noise, shadow correction, and radiative-transfer modeling uncertainties. Values above 1 occur in very bright conditions such as sun glint on the ocean, foam, haze, or thin clouds, where the measured radiance temporarily exceeds the assumed physical limits. These out-of-range values are not errors but natural artifacts of the preprocessing pipeline.

We will handle these artifacts during the preprocessing phase.